# TinyRecursiveInference: Training on Google Colab

This notebook demonstrates how to train the Tiny Recursion Model (TRM) on Google Colab with a single GPU.

**Paper:** [Less is More: Recursive Reasoning with Tiny Networks](https://arxiv.org/abs/2510.04871)

**Model Size:** ~7M parameters

**Recommended GPU:** T4 (free), A100 (Colab Pro), or V100

## What you'll learn:
- Set up the TRM training environment
- Prepare ARC-AGI datasets
- Train a small TRM model
- Evaluate and visualize results
- Export trained models

## 1. Check GPU Availability

Make sure you have GPU runtime enabled: **Runtime > Change runtime type > Hardware accelerator > GPU**

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Clone Repository and Install Dependencies

In [ ]:
# Clone the repository
!git clone https://github.com/HarleyCoops/TinyRecursiveInference.git
%cd TinyRecursiveInference

# Check what we have
!ls -la

In [ ]:
# Install dependencies
# Note: Using stable PyTorch instead of nightly for better Colab compatibility
!pip install --upgrade pip wheel setuptools
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -r requirements.txt

# Optional: Install adam-atan2 optimizer (may take a few minutes to compile)
# Uncomment if you want to use the exact optimizer from the paper
# !pip install --no-cache-dir --no-build-isolation adam-atan2

print("\n✓ Installation complete!")

## 3. Configure Experiment Tracking (Optional)

Weights & Biases is optional but recommended for tracking training progress.

In [ ]:
# Option 1: Login to Weights & Biases
import wandb

# Get your API key from https://wandb.ai/authorize
# wandb.login()  # Uncomment and run this cell to login interactively

# Option 2: Disable W&B (runs locally only)
import os
os.environ['WANDB_MODE'] = 'disabled'
print("W&B disabled - training metrics will be printed to console only")

## 4. Prepare Dataset

We'll prepare a small ARC-AGI dataset. For quick experimentation, we use fewer augmentations.

In [ ]:
# Build ARC-AGI-1 dataset with reduced augmentation for faster setup
# Full paper uses --num-aug 1000, but 100 is fine for experimentation
!python -m dataset.build_arc_dataset \
  --input-file-prefix kaggle/combined/arc-agi \
  --output-dir data/arc1-colab-aug-100 \
  --subsets training evaluation \
  --test-set-name evaluation \
  --num-aug 100

print("\n✓ Dataset prepared!")
!ls -lh data/arc1-colab-aug-100/train/

## 5. Create Training Configuration

We'll create a lightweight config optimized for single-GPU training.

In [ ]:
# Create a config file optimized for Colab
colab_config = """
# Colab-optimized training config

defaults:
  - arch: trm
  - _self_

hydra:
  output_subdir: null

# Data path
data_paths: ['data/arc1-colab-aug-100']
data_paths_test: []

evaluators:
  - name: arc@ARC

# Reduced batch size for single GPU
global_batch_size: 64

# Shorter training for quick experimentation
epochs: 10000
eval_interval: 1000
checkpoint_every_eval: True

# Same hyperparameters as paper
lr: 1e-4
lr_min_ratio: 1.0
lr_warmup_steps: 500

beta1: 0.9
beta2: 0.95
weight_decay: 0.1
puzzle_emb_weight_decay: 0.1

puzzle_emb_lr: 1e-2

seed: 0
min_eval_interval: 0

ema: False  # Disable EMA for faster training
freeze_weights: False
"""

with open('config/cfg_colab.yaml', 'w') as f:
    f.write(colab_config)

print("✓ Colab config created!")
!cat config/cfg_colab.yaml

## 6. Start Training

Now we'll train the model! This will take several hours depending on your GPU.

**Expected training times:**
- T4 GPU: ~4-6 hours for 10,000 epochs
- A100 GPU: ~1-2 hours for 10,000 epochs

The model will:
- Save checkpoints every 1000 epochs
- Print training metrics (loss, accuracy)
- Evaluate on validation set periodically

In [ ]:
# Train with the Colab config
# Note: Using --config-name to load our custom config
!python pretrain.py \
  --config-name cfg_colab \
  arch=trm \
  arch.L_layers=2 \
  arch.H_cycles=3 \
  arch.L_cycles=4 \
  +run_name="colab_experiment"

## 7. Check Training Results

In [ ]:
# List checkpoints
!ls -lh checkpoints/

# Find the latest checkpoint
import glob
import os

checkpoint_dirs = glob.glob('checkpoints/**/colab_experiment', recursive=True)
if checkpoint_dirs:
    checkpoint_dir = checkpoint_dirs[0]
    print(f"\nCheckpoint directory: {checkpoint_dir}")
    
    checkpoints = sorted(glob.glob(f"{checkpoint_dir}/step_*"))
    if checkpoints:
        print(f"\nFound {len(checkpoints)} checkpoints:")
        for ckpt in checkpoints[-5:]:  # Show last 5
            print(f"  - {ckpt}")
else:
    print("No checkpoints found yet. Training may still be in progress.")

## 8. Run Inference (Optional)

Test your trained model on a sample ARC puzzle.

In [ ]:
# Load the model and run inference
from tiny_recursive_inference import model_loader, inference
import json

# Find latest checkpoint
checkpoint_dirs = glob.glob('checkpoints/**/colab_experiment', recursive=True)
if not checkpoint_dirs:
    print("No checkpoint found. Please complete training first.")
else:
    checkpoint_dir = checkpoint_dirs[0]
    
    try:
        # Load model
        print(f"Loading model from {checkpoint_dir}...")
        model, config = model_loader.load_trm_checkpoint(checkpoint_dir)
        print("✓ Model loaded!")
        
        # Load a sample puzzle from evaluation set
        with open('kaggle/combined/arc-agi_evaluation_challenges.json') as f:
            puzzles = json.load(f)
        
        puzzle_id = list(puzzles.keys())[0]
        puzzle = puzzles[puzzle_id]
        
        print(f"\nTesting on puzzle: {puzzle_id}")
        print(f"Number of training examples: {len(puzzle['train'])}")
        print(f"Number of test examples: {len(puzzle['test'])}")
        
        # Run inference
        # Note: You would need to preprocess the puzzle first
        # This is a simplified example
        print("\n✓ Inference setup complete!")
        print("For full inference, see: tiny_recursive_inference/gradio_app.py")
        
    except Exception as e:
        print(f"Error loading model: {e}")
        print("This is normal if training just started.")

## 9. Export Results

Download your trained model and checkpoints.

In [ ]:
# Create a zip file with checkpoints
import shutil

checkpoint_dirs = glob.glob('checkpoints/**/colab_experiment', recursive=True)
if checkpoint_dirs:
    checkpoint_dir = checkpoint_dirs[0]
    
    # Create zip
    output_zip = 'trm_colab_checkpoints'
    shutil.make_archive(output_zip, 'zip', checkpoint_dir)
    
    print(f"✓ Created {output_zip}.zip")
    print(f"Size: {os.path.getsize(output_zip + '.zip') / 1e6:.2f} MB")
    
    # Download (in Colab)
    try:
        from google.colab import files
        files.download(f"{output_zip}.zip")
        print("\n✓ Download started!")
    except:
        print(f"\nTo download, right-click {output_zip}.zip in the file browser")
else:
    print("No checkpoints to export yet.")

## Next Steps

### Continue Training
- Increase `epochs` in the config for longer training
- Enable EMA for better performance: `ema=True`
- Try different model variants: `arch=trm_singlez` or `arch=trm_hier6`

### Advanced Experiments
- Train on Sudoku: Use `dataset/build_sudoku_dataset.py`
- Train on Maze: Use `dataset/build_maze_dataset.py`
- Multi-GPU training: Use Prime Intellect or HuggingFace Jobs

### Deployment
- Upload to Hugging Face: See `tiny_recursive_inference/publishers.py`
- Create Gradio demo: Run `gradio_app.py`
- Edge deployment: See `SNAPDRAGON_NPU_DEPLOYMENT.md`

### Resources
- Paper: https://arxiv.org/abs/2510.04871
- GitHub: https://github.com/HarleyCoops/TinyRecursiveInference
- Documentation: See README.md and CLAUDE.md in the repo

## Troubleshooting

### Out of Memory Errors
- Reduce `global_batch_size` to 32 or 16
- Reduce `hidden_size` to 256 or 384
- Disable EMA: `ema=False`

### Slow Training
- Check GPU is being used: `torch.cuda.is_available()`
- Reduce dataset augmentations: Use `--num-aug 50`
- Use smaller model: `arch.L_layers=1`

### Installation Issues
- Use stable PyTorch instead of nightly
- Skip adam-atan2 (PyTorch Adam works fine)
- Check CUDA version matches PyTorch: `nvcc --version`